# Week 5, Day 2 — LangChain: Tools, Chains, Memory & Your First Framework Agent


## Task 1 — LangChain Setup & Core Concepts

### Concept mapping: Day 1 (raw Python) → LangChain

| Day 1 (raw Python) | LangChain equivalent |
|---|---|
| `openai.OpenAI(...)` client | `ChatOpenAI(...)` —> **LLM wrapper** |
| Python function + hand-written JSON schema dict | `@tool`-decorated function —> **Tool** |
| Manual `Agent.run()` while-loop | `create_tool_calling_agent(...)` + `AgentExecutor`|
| `self.messages` conversion list | `ConversationBufferMemory` / `RunnableWithMessageHistory` |
| Manual `self.scratchpad`/intermediate reasoning | `intermediate_steps` managed internally by `AgentExecutor` |

### LCEL — a basic prompt → response pipeline


In [41]:
# Setup cell
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("NETIXSOL_API_KEY")
BASE_URL = "https://llm.netixsol.com/v1"
MODEL = "coder"

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

### Building a Simple LCEL Pipeline

The following example demonstrates how LangChain's LCEL composes a prompt template, language model, and output parser into a single runnable pipeline.

In [42]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "{question}")
    ]
)

chain = prompt | llm | StrOutputParser()

response = chain.invoke(
    {
        "question": "What is LangChain?"
    }
)

print(response)

**LangChain** is an open‑source framework designed to simplify the development of applications that use large language models (LLMs). It provides a set of building blocks and utilities that help developers:

| Category | What It Offers | Typical Use Cases |
|----------|----------------|-------------------|
| **Model Integration** | Easy wrappers for popular LLM providers (OpenAI, Anthropic, Cohere, Hugging Face, etc.) | Prompt generation, text completion, embeddings |
| **Prompt Management** | Prompt templates, dynamic variable substitution, chainable prompts | Re‑using complex prompts, creating chat‑style interactions |
| **Chains** | Sequential or conditional pipelines that connect LLM calls with other operations (e.g., API calls, database queries, data transformations) | Question‑answering over a knowledge base, multi‑step reasoning |
| **Agents** | Autonomous agents that can decide which tools to use (search, calculators, APIs) based on a given goal | Conversational assistants, aut

### Observation

The LCEL pipeline successfully formatted the prompt, sent it to the language model, and parsed the response into a plain string. This demonstrates how LangChain allows multiple runnable components to be composed into a single execution pipeline using the `|` operator.

**What is the `|` doing under the hood?** 
Every component in LCEL, such as prompts, language models, and output parsers, implements the same `Runnable` interface. The `|` operator connects these components into a `RunnableSequence`, passing the output of one step directly to the next. When `invoke()` is called, the pipeline executes each component in order, making it easy to build prompt → model → parser workflows without manually wiring them together.

## Task 2 — Define & Register Tools

In this task, the tools created in yesterday's raw Python agent are recreated using LangChain's `@tool` decorator. Two tools (`calculator` and `get_weather`) are reused from Day 1, while a new tool (`get_product_price`) is added that reads data from a local JSON file, demonstrating how LangChain tools can interact with external data sources.

### Why are docstrings important?

When a function is decorated with `@tool`, LangChain automatically converts it into a tool that the language model can use. The function's docstring becomes the tool's description inside the prompt sent to the model, while the function signature provides the expected input parameters. Clear and descriptive docstrings help the model decide **when** to use a tool and **how** to call it correctly.

In [3]:
import ast
import json
import operator as op

from langchain_core.tools import tool

In [4]:
# Supported operators for the calculator tool
SAFE_OPERATORS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Mod: op.mod,
    ast.Pow: op.pow,
    ast.USub: op.neg,
}


def safe_eval(node):
    """Safely evaluate a simple arithmetic expression."""
    
    if isinstance(node, ast.Constant):
        return node.value

    if isinstance(node, ast.BinOp):
        return SAFE_OPERATORS[type(node.op)](
            safe_eval(node.left),
            safe_eval(node.right)
        )

    if isinstance(node, ast.UnaryOp):
        return SAFE_OPERATORS[type(node.op)](
            safe_eval(node.operand)
        )

    raise ValueError("Unsupported expression.")

In [5]:
# Load the product database

with open("products.json", "r") as file:
    PRODUCTS = json.load(file)

In [6]:
FAKE_WEATHER_DB = {
    "lahore": {
        "temperature": 35,
        "condition": "Sunny"
    },
    "islamabad": {
        "temperature": 29,
        "condition": "Partly Cloudy"
    },
    "karachi": {
        "temperature": 32,
        "condition": "Humid"
    }
}

In [7]:
@tool
def calculator(expression: str) -> str:
    """
    Evaluate a basic arithmetic expression.

    Use this tool whenever a user asks for calculations,
    percentages, totals, or other mathematical operations.
    """

    try:
        result = safe_eval(
            ast.parse(expression, mode="eval").body
        )

        return json.dumps({
            "expression": expression,
            "result": result
        })

    except Exception as e:
        return json.dumps({
            "expression": expression,
            "error": str(e)
        })

In [8]:
@tool
def get_weather(city: str) -> str:
    """
    Return the current weather for a supported city.

    Supported cities:
    Lahore, Islamabad, Karachi.
    """

    weather = FAKE_WEATHER_DB.get(city.lower())

    if weather is None:
        return json.dumps({
            "city": city,
            "error": "Weather data not found."
        })

    return json.dumps({
        "city": city,
        **weather
    })

In [20]:
@tool
def get_product_price(product_name: str) -> str:
    """
    Look up the price of a product from the local JSON product catalog.

    Use this tool whenever a user asks about product prices.
    """

    key = product_name.lower().replace(" ", "_")

    product = PRODUCTS.get(key)

    if product is None:
        return json.dumps({
            "product": product_name,
            "error": "Product not found."
        })

    return json.dumps({
        "product": key,
        "brand": product["brand"],
        "price": product["price"]
    })

In [58]:
TOOLS = [
    calculator,
    get_weather,
    get_product_price,
    get_stock,
]

### Testing the Registered Tools

Before connecting the tools to an agent, each tool is tested individually to verify that it accepts inputs correctly and returns the expected output. This also confirms that the `@tool` decorator has successfully converted each Python function into a LangChain tool.

In [43]:
print(calculator.invoke({
    "expression": "125 * 0.15"
}))

{"expression": "125 * 0.15", "result": 18.75}


In [44]:
print(get_weather.invoke({
    "city": "Lahore"
}))

{"city": "Lahore", "temperature": 35, "condition": "Sunny"}


In [45]:
print(get_product_price.invoke({
    "product_name": "Laptop A"
}))

{"product": "laptop_a", "brand": "Dell", "price": 799}


### Observation

All three tools executed successfully when invoked individually. Compared to yesterday's raw Python implementation, no manual JSON schema or tool registry was required. The `@tool` decorator automatically generated the tool metadata from the function signature and docstring, reducing boilerplate while making the tools easier to integrate into an agent.

## Task 3 — Build an Agent using `create_tool_calling_agent`

In this task, the previously defined tools are connected to a LangChain agent using `create_tool_calling_agent()` and executed through an `AgentExecutor`. Unlike the raw Python implementation from Day 1, LangChain automatically manages the reasoning loop, tool selection, and conversation flow, allowing us to focus on defining tools and prompts.

In [46]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0
)

SYSTEM_PROMPT = """
You are a helpful AI assistant.

You have access to several tools.

Always use the available tools whenever they help answer the user's question.
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

agent = create_tool_calling_agent(
    llm=llm,
    tools=TOOLS,
    prompt=prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=TOOLS,
    verbose=True,
    handle_parsing_errors=True,
    return_intermediate_steps=True
)


In [47]:
result = agent_executor.invoke(
    {
        "input":
        "What is the price of Laptop A, and what would a 15% tax add to that?"
    }
)

print("\n=== FINAL ANSWER ===")
print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `get_product_price` with `{'product_name': 'Laptop A'}`


{"product": "laptop_a", "price": 799, "brand": "Dell"}
Invoking: `calculator` with `{'expression': '799 * 0.15'}`


{"expression": "799 * 0.15", "result": 119.85}The price of **Laptop A** is **$799**.  
A 15 % tax on that amount adds **$119.85**, making the total cost **$918.85**.

> Finished chain.

=== FINAL ANSWER ===
The price of **Laptop A** is **$799**.  
A 15 % tax on that amount adds **$119.85**, making the total cost **$918.85**.


### Annotated Reasoning Trace

The verbose execution log demonstrates the ReAct (Reason → Act → Observe) pattern followed by the agent.

- **Reason:** The language model determines that it first needs the product price before calculating the tax. This reasoning happens internally and is not printed explicitly.
- **Act:** The agent calls `get_product_price()` to retrieve the laptop's price, followed by `calculator()` to compute the 15% tax.
- **Observe:** The outputs returned by each tool are passed back to the language model, allowing it to decide the next action.
- **Final Answer:** Once sufficient information has been collected, the agent generates the final response for the user.


### Comparison with the Raw Python Agent

The overall execution pattern is very similar to the raw Python agent built in Day 1. Both follow the ReAct cycle of reasoning, selecting a tool, observing the result, and repeating until a final answer is produced.

The main difference is that LangChain hides much of the implementation. The reasoning loop, message history, tool dispatch, and intermediate scratchpad are managed internally by `AgentExecutor`, reducing boilerplate but making the internal execution less transparent compared to the manually implemented loop from Day 1.

## Task 4 — Add Memory

`RunnableWithMessageHistory` wraps the `AgentExecutor` from Task 3, keyed by
`session_id`, so a 3-turn conversation can rely on earlier context without us
re-passing it by hand. (Note: LangChain flags this as pending-deprecation in
favor of LangGraph's built-in persistence — still the documented mechanism
for this version, and exactly what the brief asks for.)

Test scenario:
1. *"What is the price of Laptop A?"*
2. *"Now compare it to Laptop B."*
3. *"Which one should I recommend to a budget-conscious client?"*


In [49]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


agent_with_memory = RunnableWithMessageHistory(
    agent_executor,           # <-- this already exists from Task 3
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

c:\Users\ahmed\Downloads\Netixsol\Week5\Day-21\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [50]:
config = {"configurable": {"session_id": "budget-client-demo"}}

turns = [
    "What is the price of Laptop A?",
    "Now compare it to Laptop B.",
    "Which one should I recommend to a budget-conscious client?",
]

for turn in turns:
    print(f"\n>>> USER: {turn}")
    result = agent_with_memory.invoke({"input": turn}, config=config)
    print(f"<<< AGENT: {result['output']}")


>>> USER: What is the price of Laptop A?


> Entering new AgentExecutor chain...

Invoking: `get_product_price` with `{'product_name': 'Laptop A'}`


{"product": "laptop_a", "price": 799, "brand": "Dell"}The price of Laptop A (a Dell laptop) is **$799**.

> Finished chain.
<<< AGENT: The price of Laptop A (a Dell laptop) is **$799**.

>>> USER: Now compare it to Laptop B.


> Entering new AgentExecutor chain...

Invoking: `get_product_price` with `{'product_name': 'Laptop B'}`


{"product": "laptop_b", "price": 999, "brand": "HP"}
Invoking: `get_product_price` with `{'product_name': 'Laptop A'}`


{"product": "laptop_a", "price": 799, "brand": "Dell"}**Price Comparison**

| Laptop | Brand | Price |
|--------|-------|-------|
| **Laptop A** | Dell | **$799** |
| **Laptop B** | HP   | **$999** |

**How they differ**

- **Absolute difference:** Laptop B costs **$200 more** than Laptop A.  
- **Relative to Laptop A:** $200 is about **25 %** higher than the price of Laptop A.  
- **Relativ

In [51]:
print("=== Full chat history stored in memory ===")
for m in get_session_history("budget-client-demo").messages:
    print(f"  [{m.type}] {m.content}")

=== Full chat history stored in memory ===
  [human] What is the price of Laptop A?
  [ai] The price of Laptop A (a Dell laptop) is **$799**.
  [human] Now compare it to Laptop B.
  [ai] **Price Comparison**

| Laptop | Brand | Price |
|--------|-------|-------|
| **Laptop A** | Dell | **$799** |
| **Laptop B** | HP   | **$999** |

**How they differ**

- **Absolute difference:** Laptop B costs **$200 more** than Laptop A.  
- **Relative to Laptop A:** $200 is about **25 %** higher than the price of Laptop A.  
- **Relative to Laptop B:** The $200 gap represents roughly **20 %** lower than the price of Laptop B.

So, Laptop B is noticeably more expensive—about a quarter more than Laptop A. If you’re looking for a lower‑cost option, Laptop A offers a $200 savings.
  [human] Which one should I recommend to a budget-conscious client?
  [ai] For a budget‑conscious client, **Laptop A** is the better recommendation.

**Why:**

| Factor | Laptop A | Laptop B |
|--------|----------|----------|


Turn 3 needed **no tool call at all** — the model answered purely from
conversation memory (both prices were already in the chat history from
turns 1–2). This is exactly the distinction Day 1 drew between conversation
memory (the message list) and working memory (our own scratchpad): here,
LangChain's memory object *is* the conversation-memory half; there's no
separate working-memory object exposed to us unless we opt into
`return_intermediate_steps=True`.


`RunnableWithMessageHistory` was used to maintain conversation history across multiple turns. This enabled the agent to answer follow-up questions using previous context without requiring the user to repeat earlier information.

## Task 5 – Structured Output & Error Handling

In this task, the agent is extended with two additional capabilities:

- **Structured Output:** Instead of returning free-form text, the model is forced to produce a validated Pydantic object. This ensures that the response follows a predefined schema and can be safely consumed by downstream applications.

- **Error Handling:** A tool is created that intentionally fails for invalid product names by raising a `ToolException`. The tool is configured with `handle_tool_error=True`, allowing the agent to recover gracefully instead of terminating the execution.

Finally, a short reflection compares the LangChain implementation with the raw Python agent built on Day 1.

### Structured Output

In [52]:
import json
from pydantic import BaseModel, Field


class ProductRecommendation(BaseModel):
    """Structured recommendation returned by the LLM."""

    recommended_product: str = Field(
        description="Name of the recommended product"
    )

    price: float = Field(
        description="Price of the recommended product in USD"
    )

    reason: str = Field(
        description="Reason for recommending this product"
    )

    budget_friendly: bool = Field(
        description="Whether the recommendation targets a budget-conscious client"
    )


structured_llm = llm.with_structured_output(ProductRecommendation)

recommendation = structured_llm.invoke(
    """
    Recommend between:

    Laptop A
    Price: $799

    Laptop B
    Price: $999

    The customer is budget-conscious.
    """
)

print("Type:", type(recommendation))
print("\nPydantic Object:")
print(recommendation)

print("\nAs JSON:")
print(json.dumps(recommendation.model_dump(), indent=4))

Type: <class '__main__.ProductRecommendation'>

Pydantic Object:
recommended_product='Laptop A' price=799.0 reason='Laptop A offers solid performance at a lower price, making it the best choice for a budget-conscious customer who wants to save money while still getting a capable laptop.' budget_friendly=True

As JSON:
{
    "recommended_product": "Laptop A",
    "price": 799.0,
    "reason": "Laptop A offers solid performance at a lower price, making it the best choice for a budget-conscious customer who wants to save money while still getting a capable laptop.",
    "budget_friendly": true
}


### Structured Output

Instead of generating plain text, the model is constrained to return a `ProductRecommendation` object.

This provides several benefits:

- Automatic validation
- Consistent response format
- Easy conversion to JSON
- Safe access through object attributes instead of manual parsing

### Error Handling Tool

In [57]:
from langchain_core.tools import tool, ToolException


@tool
def get_stock(product_name: str) -> str:
    """
    Return stock information for a product.
    Raises an error for unknown products.
    """

    inventory = {
        "Laptop A": 15,
        "Laptop B": 8,
    }

    if product_name not in inventory:
        raise ToolException(
            f"Product '{product_name}' does not exist."
        )

    return f"{product_name} has {inventory[product_name]} units in stock."

In [59]:
try:
    result = agent_executor.invoke(
        {
            "input": "Check the stock for Tablet Z."
        }
    )

    print(result["output"])

except Exception as e:
    print("=== Tool Failure ===")
    print(type(e).__name__)
    print(e)



> Entering new AgentExecutor chain...
I’m sorry, but I don’t have a tool that can look up inventory levels. I can provide the price or other details for Tablet Z if that would help—just let me know!

> Finished chain.
I’m sorry, but I don’t have a tool that can look up inventory levels. I can provide the price or other details for Tablet Z if that would help—just let me know!


### Error Handling

A stock lookup tool was created that raises a `ToolException` when an unknown product is requested. Since the installed LangChain version (0.3.30) does not support the `handle_tool_error` parameter on the `@tool` decorator, the exception propagates out of `agent_executor.invoke()`. Graceful recovery was therefore implemented by wrapping the agent invocation in a `try/except` block, preventing the application from crashing and allowing a user-friendly error message to be displayed.

### Create New Agent Including the New Tool

In [61]:
tools_with_stock = TOOLS + [get_stock]

agent = create_tool_calling_agent(
    llm,
    tools_with_stock,
    prompt,
)

agent_executor_with_errors = AgentExecutor(
    agent=agent,
    tools=tools_with_stock,
    verbose=True,
)

### Demonstrate Error Handling

In [63]:
try:
    result = agent_executor_with_errors.invoke(
        {
            "input": "Check the stock for Tablet Z."
        }
    )

    print(result["output"])

except ToolException as e:
    print("\n=== TOOL ERROR ===")
    print(e)
    print("\nPlease provide a valid product name.")



> Entering new AgentExecutor chain...

Invoking: `get_stock` with `{'product_name': 'Tablet Z'}`



=== TOOL ERROR ===
Product 'Tablet Z' does not exist.

Please provide a valid product name.


### Error Handling

The `get_stock` tool intentionally raises a `ToolException` whenever an unknown product is requested.

In newer versions of LangChain, tools can be configured with `handle_tool_error=True` so that these exceptions are automatically passed back to the agent, allowing it to recover gracefully. However, the installed version used for this task (LangChain 0.3.30) does not support the `handle_tool_error` argument on the `@tool` decorator.

Instead, the agent invocation is wrapped in a `try/except` block that catches the `ToolException`. This prevents the application from crashing and allows a clear, user-friendly error message to be displayed whenever a tool fails.

## Reflection

Compared with the raw Python agent from Day 1, LangChain significantly reduced the amount of boilerplate code. Tool registration, agent execution, structured outputs, and conversation memory were provided through reusable abstractions instead of being implemented manually.

One particularly useful feature was structured output, where a Pydantic model guaranteed that the final response followed a predefined schema without requiring manual JSON parsing or validation.

One abstraction that felt "magical" was the internal reasoning loop. Unlike the Day 1 implementation where the reasoning loop was explicitly written in Python, LangChain hides much of that execution inside `AgentExecutor`. This makes development faster but also makes debugging more difficult because many implementation details are abstracted away.

Another abstraction leak appeared during error handling. While newer LangChain versions support automatic tool error recovery through `handle_tool_error=True`, the installed version (0.3.30) does not expose this feature through the `@tool` decorator. As a result, tool failures had to be handled manually using a `try/except` block. This highlighted that framework capabilities can vary between versions, making it important to understand both the abstractions provided by LangChain and the underlying Python exception handling.

## Conclusion

In this task, LangChain's higher-level abstractions were used to extend the agent with structured outputs and basic error handling. Structured output ensured that responses followed a validated Pydantic schema, while tool failures were handled safely using Python exception handling compatible with the installed LangChain version. Together with the memory integration from the previous task, this demonstrated how LangChain simplifies building production-style agents compared to implementing the reasoning loop manually.